In [23]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

from langchain_huggingface import HuggingFaceEndpoint
from langchain_huggingface.chat_models import ChatHuggingFace
from dotenv import load_dotenv
import os

In [24]:
load_dotenv()

# Initialize HuggingFace LLM
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    huggingfacehub_api_token=os.getenv("HUGGINGFACE_API_KEY")
)

model = ChatHuggingFace(llm=llm)

In [25]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str
    evaluation: str

In [26]:
def create_outline(state: BlogState) -> BlogState:
    
    # Fetch title
    topic = state['topic']

    # call LLM to create outline
    prompt = f"Create a detailed outline for a blog post about - {topic}."
    outline = model.invoke(prompt).content

    # Update state
    state['outline'] = outline

    return state


def create_blog(state: BlogState) -> BlogState:

    # Fetch outline
    outline = state['outline']

    # call LLM to create blog content
    prompt = (
    "Convert the following outline into clean plain text with no markdown formatting. "
    "Remove all symbols such as #, *, >, -, _, and do not use bold or headings. "
    "Keep the structure readable using normal text only.\n\n"
    f"{outline}"
    )

    content = model.invoke(prompt).content

    # Update state
    state['content'] = content

    return state

def evaluate_blog(state: BlogState) -> BlogState:

    # Fetch content
    content = state['content']

    # call LLM to evaluate blog content
    prompt = (
    "Evaluate the following blog content for clarity, coherence, and engagement. "
    "Provide constructive feedback and suggest improvements if necessary.\n\n"
    f"{content}"
    )

    evaluation = model.invoke(prompt).content

    # Update state with evaluation
    state['evaluation'] = evaluation

    return state

In [27]:
graph = StateGraph(BlogState)

# Nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

# Edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()

In [28]:
initial_state: BlogState = {
    'topic': 'The Future of Artificial Intelligence in Everyday Life',
    'outline': '',
    'content': '',
    'evaluation': ''
}

final_state = workflow.invoke(initial_state)

print(final_state)

{'topic': 'The Future of Artificial Intelligence in Everyday Life', 'outline': '**Blog Post Outline**  \n*Title:* **The Future of Artificial Intelligence in Everyday Life**\n\n| # | Section | Sub‑Sections | Key Talking Points |\n|---|---------|--------------|--------------------|\n| **1** | **Hook & Thesis** | • A striking statistic or anecdote<br>• The central question: *How will AI shape the next decade of daily living?* | • Briefly mention AI’s rapid growth (e.g., GPT‑4, Tesla Autopilot, smart home assistants).<br>• Emphasize that AI is moving from “future tech” to “today’s reality.” |\n| **2** | **What Is “Everyday AI”?** | • Definition & scope<br>• Distinguishing between “AI in products” and “AI in services” | • Explain that everyday AI refers to embedded intelligence in household devices, mobile apps, workplace tools, and public services.<br>• Examples: voice‑activated assistants, predictive text, smart thermostats, recommendation engines. |\n| **3** | **Current Landscape** | • A

In [29]:
print(final_state['outline'])

**Blog Post Outline**  
*Title:* **The Future of Artificial Intelligence in Everyday Life**

| # | Section | Sub‑Sections | Key Talking Points |
|---|---------|--------------|--------------------|
| **1** | **Hook & Thesis** | • A striking statistic or anecdote<br>• The central question: *How will AI shape the next decade of daily living?* | • Briefly mention AI’s rapid growth (e.g., GPT‑4, Tesla Autopilot, smart home assistants).<br>• Emphasize that AI is moving from “future tech” to “today’s reality.” |
| **2** | **What Is “Everyday AI”?** | • Definition & scope<br>• Distinguishing between “AI in products” and “AI in services” | • Explain that everyday AI refers to embedded intelligence in household devices, mobile apps, workplace tools, and public services.<br>• Examples: voice‑activated assistants, predictive text, smart thermostats, recommendation engines. |
| **3** | **Current Landscape** | • AI in consumer tech<br>• AI in business & industry<br>• AI in public infrastructure | • 

In [ ]:
print(final_state['evaluation'])

I’m happy to help! Please paste the blog post (or the portion you’d like reviewed) here, and I’ll give you a detailed evaluation of its clarity, coherence, and engagement—along with constructive suggestions for improvement.
